In [1]:
from quickfsfunctions import *

In [2]:
# Constant Model Parameters
num_stocks = 500
pct = 0.97
years = 0.25
# Variable Model Parameters
start_date = '2024-10-01'
metric = 'all'

In [8]:
results = pd.DataFrame()
results['metric'], results['start date'], results['pct correct'], results['stocks picked'], results['Sharpe Ratio'], results['Avg Return'], results['pct values'] = [], [], [], [], [], [], []

for metric in ["all", "income_statement", "balance_sheet", "cash_flow_statement", "computed"]:
    model = XGBRegressor(objective="reg:absoluteerror", max_depth=7, n_estimators=1000, learning_rate=0.3)
    # model = train_ten_years_ranking(model, '2003-01-01', num_stocks, years, metric)
    for month in ['01', '04', '07', '10']:
        startdate = '2003' + "-" + month + "-01"
        model = train_ranking(model, startdate, num_stocks, years, metric)[0]
    for year in [str(x) for x in range(2004, 2024)]:
        for month in ['01', '04', '07', '10']:
            startdate = year + "-" + month + "-01"
            print(metric, startdate)
            return_df = predict_df(model, startdate, num_stocks, years, metric, pct=1-pct)
            # predictions, values = predict(model, startdate, num_stocks, years, metric)
            # pct_values = return_pct(predictions, values, pct=1-pct)[0]
            num_correct = 0
            for value in return_df['Return']:
                if value >= 0.03556: # Quarterly investment grade precision
                    num_correct += 1
            returns = backtest(return_df, startdate, 1)
            results.loc[len(results)] = [metric, startdate, num_correct / len(return_df['Return']), len(return_df['Return']), one_year_sharpe(returns), np.mean(return_df['Return']), return_df['Return']]
            # Retrain the model on the previous year just predicted
            model = train_ranking(model, startdate, num_stocks, years, metric)[0]
            results.to_csv("data/experiments/quickFSrankingQuarterlyPrecision+Sharpe+Return500_0.99")

all 2004-01-01
all 2004-04-01
all 2004-07-01
all 2004-10-01
all 2005-01-01
all 2005-04-01
all 2005-07-01
all 2005-10-01
all 2006-01-01
all 2006-04-01
all 2006-07-01
all 2006-10-01
all 2007-01-01
all 2007-04-01
all 2007-07-01
all 2007-10-01
all 2008-01-01
all 2008-04-01
all 2008-07-01
all 2008-10-01
all 2009-01-01
all 2009-04-01
all 2009-07-01
all 2009-10-01
all 2010-01-01
all 2010-04-01
all 2010-07-01
all 2010-10-01
all 2011-01-01
all 2011-04-01
all 2011-07-01
all 2011-10-01
all 2012-01-01
all 2012-04-01
all 2012-07-01
all 2012-10-01
all 2013-01-01
all 2013-04-01
all 2013-07-01
all 2013-10-01
all 2014-01-01
all 2014-04-01
all 2014-07-01
all 2014-10-01
all 2015-01-01
all 2015-04-01
all 2015-07-01
all 2015-10-01
all 2016-01-01
all 2016-04-01
all 2016-07-01
all 2016-10-01
all 2017-01-01
all 2017-04-01
all 2017-07-01
all 2017-10-01
all 2018-01-01
all 2018-04-01
all 2018-07-01
all 2018-10-01
all 2019-01-01
all 2019-04-01
all 2019-07-01
all 2019-10-01
all 2020-01-01
all 2020-04-01
all 2020-0

computed 2007-07-01
computed 2007-10-01
computed 2008-01-01
computed 2008-04-01
computed 2008-07-01
computed 2008-10-01
computed 2009-01-01
computed 2009-04-01
computed 2009-07-01
computed 2009-10-01
computed 2010-01-01
computed 2010-04-01
computed 2010-07-01
computed 2010-10-01
computed 2011-01-01
computed 2011-04-01
computed 2011-07-01
computed 2011-10-01
computed 2012-01-01
computed 2012-04-01
computed 2012-07-01
computed 2012-10-01
computed 2013-01-01
computed 2013-04-01
computed 2013-07-01
computed 2013-10-01
computed 2014-01-01
computed 2014-04-01
computed 2014-07-01
computed 2014-10-01
computed 2015-01-01
computed 2015-04-01
computed 2015-07-01
computed 2015-10-01
computed 2016-01-01
computed 2016-04-01
computed 2016-07-01
computed 2016-10-01
computed 2017-01-01
computed 2017-04-01
computed 2017-07-01
computed 2017-10-01
computed 2018-01-01
computed 2018-04-01
computed 2018-07-01
computed 2018-10-01
computed 2019-01-01
computed 2019-04-01
computed 2019-07-01
computed 2019-10-01


In [12]:
results = pd.read_csv("data/experiments/quickFSrankingQuarterlyPrecision+Sharpe+Return2500_0.99", index_col=0)
# Yearly: data/experiments/quickFSrankingPrecision+Sharpe+Return2500_0.97
print("Precision:")
for metric in ["all", "income_statement", "balance_sheet", "cash_flow_statement", "computed"]:
    print(metric, np.mean(results[results['metric'] == metric].iloc[:, 2]))
print("Sharpe:")
for metric in ["all", "income_statement", "balance_sheet", "cash_flow_statement", "computed"]:
    print(metric, np.mean(results[results['metric'] == metric].iloc[:, 4]))
print("Return:")
for metric in ["all", "income_statement", "balance_sheet", "cash_flow_statement", "computed"]:
    print(metric, np.mean(results[results['metric'] == metric].iloc[:, 5]))

Precision:
all 0.4958310908862379
income_statement 0.5017739572518984
balance_sheet 0.4901087219561484
cash_flow_statement 0.4933629768923886
computed 0.5140951664756812
Sharpe:
all 0.6938937298102935
income_statement 0.7431596876040656
balance_sheet 0.641006957169414
cash_flow_statement 0.6376004856784391
computed 0.6782776873558087
Return:
all 0.06978024175713884
income_statement 0.06817868809448475
balance_sheet 0.0656391922460404
cash_flow_statement 0.05442556913873797
computed 0.07580993546613707


In [13]:
results = pd.read_csv("data/experiments/quickFSrankingQuarterlyPrecision+Sharpe+Return2500_0.99", index_col=0)
total = 1
metric = 'computed'
for value in results[results['metric'] == metric]['Avg Return']:
    total *= 1 + value
print(total**(1/20)-1)

0.27140297285654813


In [10]:
income_mean = np.mean(results[results['metric'] == 'income_statement']['pct correct'])
all_mean = np.mean(results[results['metric'] == 'all']['pct correct'])
total = np.sum(results[results['metric'] == 'all']['stocks picked'])
print(f"z-score: {(income_mean - all_mean) / (np.sqrt(income_mean * (1 - income_mean) + all_mean * (1 - all_mean)) / np.sqrt(total))}")

z-score: 0.26617688834102243


In [11]:
comp_mean = np.mean(results[results['metric'] == 'computed'].iloc[:, 5])
comp_std = np.std(results[results['metric'] == 'computed'].iloc[:, 5])
other_mean = np.mean(results[results['metric'] == 'cash_flow_statement'].iloc[:, 5])
other_std = np.std(results[results['metric'] == 'cash_flow_statement'].iloc[:, 5])
print(f"Z-score: {(comp_mean - other_mean) / (np.sqrt(other_std**2 + comp_std**2) / np.sqrt(len(results) / 5))}")

Z-score: 0.8608559313100823


In [18]:
results = pd.read_csv("data/experiments/investmentGradePrecision2500_0.98", index_col=0)
for metric in ["balancegrowth", "cashflowgrowth", "incomegrowth", "keymetrics", "compiled"]:
    print(metric, np.mean(results[results['metric'] == metric].iloc[:, 2]))
display(results)

balancegrowth 0.4708843537414966
cashflowgrowth 0.41768707482993195
incomegrowth 0.4518367346938776
keymetrics 0.5729251700680272
compiled 0.5034013605442177


,metric,start date,pct correct
0,balancegrowth,2020-12-01,0.285714
1,balancegrowth,2021-12-01,0.346939
2,balancegrowth,2022-12-01,0.780000
3,cashflowgrowth,2020-12-01,0.408163
4,cashflowgrowth,2021-12-01,0.244898
5,cashflowgrowth,2022-12-01,0.600000
6,incomegrowth,2020-12-01,0.367347
7,incomegrowth,2021-12-01,0.408163
8,incomegrowth,2022-12-01,0.580000
9,keymetrics,2020-12-01,0.428571


In [73]:
results = pd.DataFrame()
results['metric'], results['start date'], results['avg feature importance'], results['# in top 10'] = [], [], [], []
metric_data = pd.read_csv("data/metric_metadata.csv", index_col=0)
metric = 'all'

model = XGBRegressor(objective="reg:absoluteerror", max_depth=7, n_estimators=1000, learning_rate=0.3)
# model = train_ten_years_ranking(model, '2003-01-01', num_stocks, years, metric)
for month in ['01', '04', '07', '10']:
    startdate = '2003' + "-" + month + "-01"
    model = train_ranking(model, startdate, num_stocks, years, metric)[0]
for year in [str(x) for x in range(2004, 2024)]:
    for month in ['01', '04', '07', '10']:
        startdate = year + "-" + month + "-01"
        print(metric, startdate)
        redundant_model = XGBRegressor(max_depth=3, n_estimators=100, learning_rate=1)
        redundant_model, xlist = train_ranking(redundant_model, startdate, num_stocks, years, metric)
        feature_df = feature_importance(model, xlist)
        indexed_df = feature_df.reset_index()
        for metric_type in ["income_statement", "balance_sheet", "cash_flow_statement", "computed"]:
            feature_scores = []
            num_top_ten = 0
            for indiv_metric in metric_data[metric_data['statement_type'] == metric_type]['metric']:
                if indiv_metric in feature_df.index:
                    feature_scores.append(indexed_df[indexed_df['index'] == indiv_metric].iloc[0, 1])
                    if indexed_df[indexed_df['index'] == indiv_metric].index[0] < 10:
                        num_top_ten += 1
            results.loc[len(results)] = [metric_type, startdate, np.mean(feature_scores), num_top_ten]
        # Retrain the model on the previous year just predicted
        model = train_ranking(model, startdate, num_stocks, years, metric)[0]
        results.to_csv("data/experiments/featureImportance.csv")

all 2004-01-01
all 2004-04-01
all 2004-07-01
all 2004-10-01
all 2005-01-01
all 2005-04-01
all 2005-07-01
all 2005-10-01
all 2006-01-01
all 2006-04-01
all 2006-07-01
all 2006-10-01
all 2007-01-01
all 2007-04-01
all 2007-07-01
all 2007-10-01
all 2008-01-01
all 2008-04-01
all 2008-07-01
all 2008-10-01
all 2009-01-01
all 2009-04-01
all 2009-07-01
all 2009-10-01
all 2010-01-01
all 2010-04-01
all 2010-07-01
all 2010-10-01
all 2011-01-01
all 2011-04-01
all 2011-07-01
all 2011-10-01
all 2012-01-01
all 2012-04-01
all 2012-07-01
all 2012-10-01
all 2013-01-01
all 2013-04-01
all 2013-07-01
all 2013-10-01
all 2014-01-01
all 2014-04-01
all 2014-07-01
all 2014-10-01
all 2015-01-01
all 2015-04-01
all 2015-07-01
all 2015-10-01
all 2016-01-01
all 2016-04-01
all 2016-07-01
all 2016-10-01
all 2017-01-01
all 2017-04-01
all 2017-07-01
all 2017-10-01
all 2018-01-01
all 2018-04-01
all 2018-07-01
all 2018-10-01
all 2019-01-01
all 2019-04-01
all 2019-07-01
all 2019-10-01
all 2020-01-01
all 2020-04-01
all 2020-0

In [15]:
results = pd.DataFrame()
results['metric'], results['start date'], results['feature importance'] = [], [], []
metric_data = pd.read_csv("data/metric_metadata.csv", index_col=0)
metric = 'all'

model = XGBRegressor(objective="reg:absoluteerror", max_depth=7, n_estimators=1000, learning_rate=0.3)
# model = train_ten_years_ranking(model, '2003-01-01', num_stocks, years, metric)
for month in ['01', '04', '07', '10']:
    startdate = '2003' + "-" + month + "-01"
    model = train_ranking(model, startdate, num_stocks, years, metric)[0]
for year in [str(x) for x in range(2004, 2024)]:
    for month in ['01', '04', '07', '10']:
        startdate = year + "-" + month + "-01"
        print(metric, startdate)
        redundant_model = XGBRegressor(max_depth=3, n_estimators=100, learning_rate=1)
        redundant_model, xlist = train_ranking(redundant_model, startdate, num_stocks, years, metric)
        feature_df = feature_importance(model, xlist)
        indexed_df = feature_df.reset_index()
        for indiv_metric in metric_data['metric']:
            if indiv_metric in feature_df.index:
                results.loc[len(results)] = [indiv_metric, startdate, indexed_df[indexed_df['index'] == indiv_metric].iloc[0, 1]]
        # Retrain the model on the previous year just predicted
        model = train_ranking(model, startdate, num_stocks, years, metric)[0]
        results.to_csv("data/experiments/featureImportanceAllFeatures.csv")

all 2004-01-01
all 2004-04-01
all 2004-07-01
all 2004-10-01
all 2005-01-01
all 2005-04-01
all 2005-07-01
all 2005-10-01
all 2006-01-01
all 2006-04-01
all 2006-07-01
all 2006-10-01
all 2007-01-01
all 2007-04-01
all 2007-07-01
all 2007-10-01
all 2008-01-01
all 2008-04-01
all 2008-07-01
all 2008-10-01
all 2009-01-01
all 2009-04-01
all 2009-07-01
all 2009-10-01
all 2010-01-01
all 2010-04-01
all 2010-07-01
all 2010-10-01
all 2011-01-01
all 2011-04-01
all 2011-07-01
all 2011-10-01
all 2012-01-01
all 2012-04-01
all 2012-07-01
all 2012-10-01
all 2013-01-01
all 2013-04-01
all 2013-07-01
all 2013-10-01
all 2014-01-01
all 2014-04-01
all 2014-07-01
all 2014-10-01
all 2015-01-01
all 2015-04-01
all 2015-07-01
all 2015-10-01
all 2016-01-01
all 2016-04-01
all 2016-07-01
all 2016-10-01
all 2017-01-01
all 2017-04-01
all 2017-07-01
all 2017-10-01
all 2018-01-01
all 2018-04-01
all 2018-07-01
all 2018-10-01
all 2019-01-01
all 2019-04-01
all 2019-07-01
all 2019-10-01
all 2020-01-01
all 2020-04-01
all 2020-0